# Module 2: Tree Objects

Congratulations on building a working blob store! You now have a content-addressable storage system that can store and retrieve any file's content efficiently and safely.

But blobs alone only solve half the problem. A blob stores the **content** of a file, but it says nothing about:
- The file's **name**
- Where the file lives in the **directory hierarchy**
- What other files exist alongside it

This is where **Tree Objects** come in. In Git, a tree object is like a directory snapshot: it stores a list of file names and their corresponding blob hashes, plus references to subdirectories (which are themselves tree objects).

With trees, we can capture the entire structure of a project at a point in time.

## 1. The Problem with Blobs Alone

Imagine you have two files:
- `src/main.py` with content "print('hello')"
- `README.md` with content "# My Project"

Your blob store would contain two blobs:
- `blob_A`: hash of `print('hello')`
- `blob_B`: hash of `# My Project`

But how do you know:
- Which blob corresponds to which filename?
- That `main.py` is inside a `src/` directory?

You need a way to map **names** to **hashes** and preserve the **directory structure**.

## 2. What is a Tree Object?

A tree object stores a list of **entries**. Each entry represents either:
1. A **file** (blob): stores the file mode, filename, and blob hash
2. A **directory** (tree): stores the file mode, directory name, and tree hash

### Tree Entry Format (Git's format):
`<mode> <type> <name>\0<hash>`
- `<mode>`: File permissions (e.g., `100644` for regular file, `100755` for executable, `040000` for directory)
- `<type>`: In Git's internal format, this is implicit based on mode, but we'll make it explicit
- `<name>`: The filename or directory name
- `\0`: Null byte separator
- `<hash>`: The SHA-1 hash (as 20 raw bytes, not hex)

### Example Tree:
If we have:
- `src/` (directory) → hash `tree1`
- `README.md` (file) → hash `blob_B`

And inside `src/`:
- `main.py` (file) → hash `blob_A`

Then the root tree would contain two entries:
1. `040000 tree src\0<20-byte hash of tree1>`
2. `100644 blob README.md\0<20-byte hash of blob_B>`

And the `src` tree would contain:
1. `100644 blob main.py\0<20-byte hash of blob_A>`

**Important**: Tree entries are sorted alphabetically by name. This ensures that identical trees always produce the same hash, enabling deduplication.

## 3. Implementing Tree Objects

Let's extend our VCS to handle tree objects. We'll need:
1. A way to create a tree from a dictionary of {name: hash}
2. A way to parse a tree back into that dictionary
3. Methods to store and retrieve trees (similar to blobs)
4. Handling for both files (blobs) and directories (trees)

We'll keep things simple:
- Use mode `100644` for files (blobs)
- Use mode `040000` for directories (trees)
- Store the hash as hex for readability (unlike Git's raw bytes)
- Use JSON-like structure internally for ease of understanding

### Simple Tree Representation
We'll represent a tree as a dictionary:
```python
{
    'README.md': ('blob', 'abc123...'),
    'src': ('tree', 'def456...')
}
```
Where the tuple is `(object_type, object_hash)`.

In [ ]:
import os
import hashlib
import json

class Tree:
    def __init__(self, entries=None):
        """Initialize a tree with optional entries."""
        self.entries = entries or {}  # {name: (obj_type, obj_hash)}

    def add(self, name, obj_type, obj_hash):
        """Add an entry to the tree."""
        self.entries[name] = (obj_type, obj_hash)

    def serialize(self) -> bytes:
        """Convert the tree to bytes for storage."""
        # Sort entries by name for consistent hashing
        sorted_items = sorted(self.entries.items())
        
        # Build the byte stream
        result = b""
        for name, (obj_type, obj_hash) in sorted_items:
            # Determine mode
            if obj_type == 'blob':
                mode = b'100644'
            elif obj_type == 'tree':
                mode = b'040000'
            else:
                raise ValueError(f"Unknown object type: {obj_type}")
            
            # Format: mode <space> type <space> name <null> <hash_bytes>
            # Note: We'll store hash as hex for simplicity, then convert to bytes
            # In real Git, hash is 20 raw bytes
            hash_bytes = bytes.fromhex(obj_hash)  # Convert hex to bytes
            result += mode + b' ' + obj_type.encode() + b' ' + name.encode() + b'\x00' + hash_bytes
        
        return result

    @classmethod
    def deserialize(cls, data: bytes):
        """Parse bytes back into a Tree object."""
        tree = cls()
        i = 0
        while i < len(data):
            # Find next null byte (separator)
            null_pos = data.find(b'\x00', i)
            if null_pos == -1:
                break
            
            # Extract the header: mode type name
            header = data[i:null_pos].decode()
            mode, obj_type, name = header.split(' ')
            
            # Extract the hash (20 bytes after null)
            hash_start = null_pos + 1
            hash_end = hash_start + 20
            hash_bytes = data[hash_start:hash_end]
            obj_hash = hash_bytes.hex()  # Convert back to hex
            
            # Add entry
            tree.add(name, obj_type, obj_hash)
            
            # Move pointer past this entry
            i = hash_end
        
        return tree

    def __repr__(self):
        return f"Tree({self.entries})"

# Let's test it
if __name__ == "__main__":
    # Create a tree
    tree = Tree()
    tree.add('README.md', 'blob', 'abc123def456'*8)  # 40-char hash
    tree.add('src', 'tree', 'def456ghi789'*8)
    
    print("Original tree:", tree)
    
    # Serialize
    data = tree.serialize()
    print("Serialized length:", len(data), "bytes")
    
    # Deserialize
    tree2 = Tree.deserialize(data)
    print("Deserialized tree:", tree2)
    print("Trees equal?", tree.entries == tree2.entries)
    
    # Test hash consistency
    tree3 = Tree()
    tree3.add('src', 'tree', 'def456ghi789'*8)
    tree3.add('README.md', 'blob', 'abc123def456'*8)
    
    data3 = tree3.serialize()
    print("\nSame content, different order:")
    print("Tree 2 entries:", tree2.entries)
    print("Tree 3 entries:", tree3.entries)
    print("Serialized data equal?", data == data3)
    print("(Should be True due to sorting)")

## 4. Integrating Trees into Our VCS

Now let's add tree support to our VersionControlSystem class. We'll need:

### New Methods:
1. `store_tree(tree: Tree) -> str`
   - Serializes the tree, compresses it, hashes it, and stores it
   - Returns the tree's hash
2. `retrieve_tree(tree_hash: str) -> Tree`
   - Verifies, decompresses, and deserializes the tree data
   - Returns the Tree object
3. `validate_tree(tree_hash: str) -> bool`
   - Checks if a tree object is intact

We'll follow the same pattern as our blob methods:
- Store: serialize → hash → compress → write
 - Retrieve: read → decompress → validate → deserialize
 - Validate: read → decompress → re-hash → compare

### Handling Mixed Content
Remember: a tree can contain both blobs (files) and trees (subdirectories).
When we store a tree, we don't store the actual file/directory content—just the hashes.
The actual content is stored in the blob/tree objects themselves.

In [ ]:
import os
import hashlib
import zlib

# Assuming our Tree class from above is available
# In practice, you'd import it or define it here

class VersionControlSystem:
    def __init__(self, root_directory: str = '.git'):
        self.root_directory = root_directory
        self.objects_directory = os.path.join(self.root_directory, 'objects')
        self.chunk_size = 8192
        os.makedirs(self.objects_directory, exist_ok=True)

    # ... (keep your existing blob methods: store_file_blob, validate_file_blob, etc.) ...

    def store_tree(self, tree) -> str:
        """Store a Tree object and return its hash."""
        # 1. Serialize the tree to bytes
        tree_data = tree.serialize()
        
        # 2. Calculate hash of the UNCOMPRESSED data (like Git does)
        tree_hash = hashlib.sha1(tree_data).hexdigest()
        
        # 3. Define storage path
        obj_path = os.path.join(self.objects_directory, tree_hash)
        
        # 4. Store if it doesn't exist (deduplication)
        if not os.path.exists(obj_path):
            # Compress the serialized data
            compressed = zlib.compress(tree_data)
            with open(obj_path, 'wb') as f:
                f.write(compressed)
        
        return tree_hash

    def validate_tree(self, tree_hash: str) -> bool:
        """Verify a tree object's integrity."""
        obj_path = os.path.join(self.objects_directory, tree_hash)
        if not os.path.exists(obj_path):
            raise FileNotFoundError(f"Tree object {tree_hash} not found")
        
        # Read and decompress
        with open(obj_path, 'rb') as f:
            compressed_data = f.read()
        
        try:
            tree_data = zlib.decompress(compressed_data)
        except:
            return False
        
        # Recalculate hash and compare
        calculated_hash = hashlib.sha1(tree_data).hexdigest()
        return tree_hash == calculated_hash

    def retrieve_tree(self, tree_hash: str):
        """Retrieve and return a Tree object."""
        if not self.validate_tree(tree_hash):
            raise ValueError(f"Tree object {tree_hash} is corrupted")
        
        obj_path = os.path.join(self.objects_directory, tree_hash)
        with open(obj_path, 'rb') as f:
            compressed_data = f.read()
        
        tree_data = zlib.decompress(compressed_data)
        return Tree.deserialize(tree_data)

# Example usage
if __name__ == "__main__":
    vcs = VersionControlSystem()
    
    # First, store some blobs (simulating files)
    blob1 = vcs.store_file_blob('file1.txt')  # Assume this exists
    blob2 = vcs.store_file_blob('file2.txt')
    
    # Create a tree
    tree = Tree()
    tree.add('file1.txt', 'blob', blob1)
    tree.add('file2.txt', 'blob', blob2)
    tree.add('subdir', 'tree', 'dummy_tree_hash')  # We'll ignore this for now
    
    print("Created tree:", tree)
    
    # Store the tree
    tree_hash = vcs.store_tree(tree)
    print(f"Stored tree hash: {tree_hash}")
    
    # Retrieve it
    retrieved_tree = vcs.retrieve_tree(tree_hash)
    print(f"Retrieved tree: {retrieved_tree}")
    print(f"Trees match: {tree.entries == retrieved_tree.entries}")

## 5. Putting It All Together: Snapshots

Now we have:
- **Blobs**: Store file content
- **Trees**: Store directory structure (names → hashes)

The next logical step is to create a **snapshot** of our entire working directory. This is what Git calls a "commit"—but we'll get to that in Module 3.

For now, imagine we want to save the state of our project. We would:
1. Walk through our working directory
2. For each file, call `store_file_blob` to get its blob hash
3. Build Tree objects representing directories
4. Store the top-level tree
5. The hash of that top-level tree represents our entire project snapshot!

### Why This is Powerful
- **Content Addressable**: If two projects have identical structure and content, they'll have the same tree hash.
- **Efficient**: If only one file changes, we only need to create one new blob and update the trees along its path.
- **Integrity**: Hashes guarantee we can detect any corruption.

In the next module, we'll learn how to chain these snapshots together using **Commit Objects** to create a full version history.

## 🛠️ Challenges: Build Your Tree Support

Now it's your turn to implement tree objects in your VCS! Here are progressive challenges:

### Challenge 1: Basic Tree Storage
Implement the `store_tree`, `validate_tree`, and `retrieve_tree` methods in your VersionControlSystem class.

### Challenge 2: Tree from Directory
Write a method `tree_from_directory(self, dir_path: str) -> str` that:
- Walks the directory at `dir_path`
- For each file, calls `store_file_blob` to get its blob hash
- Builds Tree objects for each directory (bottom-up)
- Returns the hash of the top-level tree

### Challenge 3: Apply Your Tree
Write a method `apply_tree(self, tree_hash: str, dest_path: str)` that:
- Retrieves the tree
- For each entry:
  - If it's a blob: retrieves the blob content and writes it to `dest_path/name`
  - If it's a tree: recursively creates the subdirectory and applies it
- Recreates the directory structure from a tree hash

### Challenge 4: The Full Snapshot
Combine Challenges 2 and 3:
1. Create a method `snapshot(self, dir_path: str) -> str` that returns the tree hash of `dir_path`
2. Create a method `restore(self, tree_hash: str, dest_path: str)` that recreates `dir_path` at `dest_path`
3. Verify that `restore(snapshot(dir), dest)` gives you an identical directory

### 💡 Hints
- Remember to sort directory entries alphabetically when building trees
- Use `os.walk()` or `os.scandir()` for directory traversal
- Handle edge cases: empty directories, special permissions, etc.
- Test with a known directory structure first

## 📝 Summary Questions

1. Why must tree entries be sorted alphabetically by name?
2. What happens to the tree hash if you change just one character in a file deep in the directory tree?
3. How does storing trees enable Git to store directory snapshots efficiently?
4. If two directories have the same files and subdirectories but in different order, will they have the same tree hash? Why or why not?

Once you've implemented your tree support, test it thoroughly! When you're ready, we'll move on to **Module 3: Commit Objects** where we'll learn how to chain snapshots together to create a complete version history.